<a href="https://colab.research.google.com/github/nataliakorczynska-cmyk/MEDICA/blob/main/L7_Korczynska_35793_kol1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Obliczam swoje id_20 (wynik: 4)

In [ ]:

print("Natalia")
print("Korczyńska")
print("35793")
print("id_20 = 4")

Natalia
Korczyńska
35793
id_20 = 4


instaluję PySpark

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, stddev, count

tworzę sesję z nazwaną moim nazwiskiem

In [ ]:
spark = SparkSession.builder.appName("Korczyńska").getOrCreate()

importuję pliki csv oraz definiuję odpowiedni separator

In [ ]:
df_wykonanie = spark.read.csv("execution.csv", header=True, inferSchema=True, sep=';')
df_osoba = spark.read.csv("person.csv", header=True, inferSchema=True, sep=';')
df_dzial = spark.read.csv("dept.csv", header=True, inferSchema=True, sep=';')
df_stanowisko = spark.read.csv("position.csv", header=True, inferSchema=True, sep=';')
df_czynnosc = spark.read.csv("action.csv", header=True, inferSchema=True, sep=';')

wyświetlam po pięć wierszy z utworzonego DataFrame

In [ ]:
df_wykonanie.show(5)

df_osoba.show(5)

df_dzial.show(5)

df_stanowisko.show(5)

df_czynnosc.show(5)

+---+-----+---------+-------+-----------+------------+----+----+-------+
| id|id_20|id_person|id_dept|id_position|id_execution|time|cost|payment|
+---+-----+---------+-------+-----------+------------+----+----+-------+
|  1|    0|        5|      4|          1|           3| 590| 487|   1384|
|  2|    1|        6|      2|          3|           1| 563| 873|    455|
|  3|    2|        1|      1|          3|           3| 160| 659|   1009|
|  4|    3|        4|      6|          1|           2| 101| 848|   1850|
|  5|    4|        1|      2|          1|           4| 476| 750|   1401|
+---+-----+---------+-------+-----------+------------+----+----+-------+
only showing top 5 rows
+---+-----+
| id| name|
+---+-----+
|  1|  Ela|
|  2|  Jan|
|  3|  Ola|
|  4|Marek|
|  5|  Ula|
+---+-----+
only showing top 5 rows
+---+---------+
| id|     dept|
+---+---------+
|  1|       IT|
|  2|  Service|
|  3|  Testing|
|  4|Technical|
|  5|   Coding|
+---+---------+
only showing top 5 rows
+---+-----------+
|

łączę główną tabelę execution z czterema tabelami słownikowymi dopasowując je po id

In [ ]:
df_joined = df_wykonanie.join(df_osoba, df_wykonanie.id_person == df_osoba.id) \
    .join(df_dzial, df_wykonanie.id_dept == df_dzial.id) \
    .join(df_stanowisko, df_wykonanie.id_position == df_stanowisko.id) \
    .join(df_czynnosc, df_wykonanie.id_execution == df_czynnosc.id) \
    .select(
        df_wykonanie.id_20,
        df_wykonanie.time.alias("czas"),
        df_wykonanie.cost.alias("koszt"),
        df_wykonanie.payment.alias("zaplata"),
        df_osoba.name.alias("imie"),
        df_dzial.dept.alias("dzial"),
        df_stanowisko.position.alias("stanowisko"),
        df_czynnosc.action.alias("czynnosc")
    )

filtruję połączony widok, aby zostawić rekordy przypisane do mojego id_20

In [ ]:
df_filtered = df_joined.filter(col("id_20") == 4)

In [ ]:
df_filtered.show(5)

+-----+----+-----+-------+------+---------+-----------+------------+
|id_20|czas|koszt|zaplata|  imie|    dzial| stanowisko|    czynnosc|
+-----+----+-----+-------+------+---------+-----------+------------+
|    4| 476|  750|   1401|   Ela|  Service|    manager|     testing|
|    4| 427|  377|    948|   Ula|   Coding|coordinator|    planning|
|    4| 188|  813|    796|  Olek|       IT|coordinator|organization|
|    4| 387|  485|   1661|Franek|  Graphic|coordinator|    planning|
|    4| 225|  173|   1612|   Ula|Technical|    manager|    planning|
+-----+----+-----+-------+------+---------+-----------+------------+
only showing top 5 rows


obliczam ilość obserwacji, średnie oraz odchylenia standardowe dla czasu, kosztu i zapłaty oraz przentuje statystyki

In [ ]:
stats = df_filtered.select(
    count("*").alias("ilosc_obserwacji"),
    mean("czas").alias("srednia_czas"),
    stddev("czas").alias("odchylenie_czas"),
    mean("koszt").alias("srednia_koszt"),
    stddev("koszt").alias("odchylenie_koszt"),
    mean("zaplata").alias("srednia_zaplata"),
    stddev("zaplata").alias("odchylenie_zaplata")
)

In [ ]:
stats.show()

+----------------+------------+------------------+-----------------+------------------+---------------+------------------+
|ilosc_obserwacji|srednia_czas|   odchylenie_czas|    srednia_koszt|  odchylenie_koszt|srednia_zaplata|odchylenie_zaplata|
+----------------+------------+------------------+-----------------+------------------+---------------+------------------+
|              30|       316.7|161.01277524812573|572.2333333333333|242.75199254600875|         1204.1|468.28552666376373|
+----------------+------------+------------------+-----------------+------------------+---------------+------------------+

